# SMS Spam Classifier — Naive Bayes
End-to-end pipeline: download → preprocess → train → evaluate → save

## 1. Download & Extract Dataset

In [ ]:
import requests
import zipfile
import io
import os

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
response = requests.get(url)

if response.status_code == 200:
    print("Download successful")
    with zipfile.ZipFile(io.BytesIO(response.content)) as z:
        z.extractall("sms_spam_collection")
        print("Extraction successful")
else:
    print("Download failed:", response.status_code)

print("Extracted files:", os.listdir("sms_spam_collection"))

## 2. Load & Inspect Dataset

In [ ]:
import pandas as pd

df = pd.read_csv(
    "sms_spam_collection/SMSSpamCollection",
    sep="\t",
    header=None,
    names=["label", "message"]
)

print("=== HEAD ===")
print(df.head())
print("\n=== DESCRIBE ===")
print(df.describe())
print("\n=== INFO ===")
print(df.info())
print("\nMissing values:\n", df.isnull().sum())
print("Duplicate entries:", df.duplicated().sum())

df = df.drop_duplicates()
print("Shape after dedup:", df.shape)

## 3. Download NLTK Resources

In [ ]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

## 4. Text Preprocessing Pipeline

In [ ]:
import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()

def preprocess_message(message):
    # Lowercase
    message = message.lower()
    # Remove punctuation/numbers, keep $ and !
    message = re.sub(r"[^a-z\s$!]", "", message)
    # Tokenize
    tokens = word_tokenize(message)
    # Remove stop words
    tokens = [w for w in tokens if w not in stop_words]
    # Stem
    tokens = [stemmer.stem(w) for w in tokens]
    # Rejoin
    return " ".join(tokens)

print("=== BEFORE PREPROCESSING ===")
print(df["message"].head())

df["message"] = df["message"].apply(preprocess_message)

print("\n=== AFTER PREPROCESSING ===")
print(df["message"].head())

## 5. Feature Extraction (Bag of Words)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# Unigrams + bigrams, filter very common/rare terms
vectorizer = CountVectorizer(min_df=1, max_df=0.9, ngram_range=(1, 2))

# Features and labels
X = vectorizer.fit_transform(df["message"])
y = df["label"].apply(lambda x: 1 if x == "spam" else 0)

print("Feature matrix shape:", X.shape)
print("Label distribution:\n", y.value_counts())

## 6. Train with Pipeline + GridSearchCV

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

pipeline = Pipeline([
    ("vectorizer", vectorizer),
    ("classifier", MultinomialNB())
])

# Tune alpha — smoothing factor for unseen words
param_grid = {
    "classifier__alpha": [0.01, 0.1, 0.15, 0.2, 0.25, 0.5, 0.75, 1.0]
}

grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring="f1")
grid_search.fit(df["message"], y)

best_model = grid_search.best_estimator_
print("Best parameters:", grid_search.best_params_)
print("Best F1 score:", round(grid_search.best_score_, 4))

## 7. Evaluate on New Messages

In [ ]:
new_messages = [
    "Congratulations! You've won a $1000 Walmart gift card. Go to http://bit.ly/1234 to claim now.",
    "Hey, are we still meeting up for lunch today?",
    "Urgent! Your account has been compromised. Verify your details here: www.fakebank.com/verify",
    "Reminder: Your appointment is scheduled for tomorrow at 10am.",
    "FREE entry in a weekly competition to win an iPad. Just text WIN to 80085 now!",
]

# Preprocess identically to training data
processed_messages = [preprocess_message(msg) for msg in new_messages]

# Vectorize and predict
X_new = best_model.named_steps["vectorizer"].transform(processed_messages)
predictions = best_model.named_steps["classifier"].predict(X_new)
probabilities = best_model.named_steps["classifier"].predict_proba(X_new)

print("-" * 60)
for i, msg in enumerate(new_messages):
    label = "SPAM" if predictions[i] == 1 else "NOT SPAM"
    spam_prob = probabilities[i][1]
    ham_prob = probabilities[i][0]
    print(f"Message : {msg}")
    print(f"Prediction      : {label}")
    print(f"Spam Probability: {spam_prob:.2f}")
    print(f"Ham  Probability: {ham_prob:.2f}")
    print("-" * 60)

## 8. Save & Load Model

In [ ]:
import joblib

model_filename = "spam_detection_model.joblib"
joblib.dump(best_model, model_filename)
print(f"Model saved to {model_filename}")

# Reload and verify
loaded_model = joblib.load(model_filename)
test_processed = [preprocess_message(msg) for msg in new_messages]
test_preds = loaded_model.predict(test_processed)
print("Loaded model predictions:", ["SPAM" if p == 1 else "HAM" for p in test_preds])